In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

import pandas as pd
import src.utils.pdata_io as pdio

from src.qc.qc_events import load_behavior_qc_tables
from src.proc.extract_epoch_windows import (
    build_prepost_epoch_windows,
    save_epoch_windows,
    load_epoch_windows,
)

data_root, pdata_root, cc_data = pdio.load_project_context()

events_df, session_summary_df, _ = load_behavior_qc_tables(
    pdata_root=pdata_root,
    filename="behavior_QC.h5"
)

/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
[LOADED] Behavior QC tables: /mnt/pdata/Classical_Conditioning/_cache/behavior_QC.h5


In [2]:
windows_prepost_1s = build_prepost_epoch_windows(
    events_df,
    window_s=1.0
)

windows_prepost_1s.head()

,animal,date,phase,event_number,anchor_name,anchor_time_s,window_position,window_s,epoch_name,window_start_s,window_end_s,start_idx,end_idx,parent_event_valid,valid_window,invalid_reason
0,NML_04,2026_01_12,air_training,0,air_off,4.8704,post,1.0,air_off_post_1s,4.8704,5.8704,NaN,NaN,False,False,parent_event_invalid
1,NML_04,2026_01_12,air_training,0,air_off,4.8704,pre,1.0,air_off_pre_1s,3.8704,4.8704,NaN,NaN,False,False,parent_event_invalid
2,NML_04,2026_01_12,air_training,0,air_on,0.0000,post,1.0,air_on_post_1s,0.0000,1.0000,NaN,NaN,False,False,parent_event_invalid
3,NML_04,2026_01_12,air_training,0,air_on,0.0000,pre,1.0,air_on_pre_1s,-1.0000,0.0000,NaN,NaN,False,False,parent_event_invalid
4,NML_04,2026_01_12,air_training,1,air_off,23.9790,post,1.0,air_off_post_1s,23.9790,24.9790,119895.0,124895.0,True,True,


In [3]:
windows_prepost_1s.groupby(
    ["phase", "epoch_name", "valid_window"]
).size().reset_index(name="n")

,phase,epoch_name,valid_window,n
0,air_training,air_off_post_1s,False,156
1,air_training,air_off_post_1s,True,3380
2,air_training,air_off_pre_1s,False,156
3,air_training,air_off_pre_1s,True,3380
4,air_training,air_on_post_1s,False,156
5,air_training,air_on_post_1s,True,3380
6,air_training,air_on_pre_1s,False,156
7,air_training,air_on_pre_1s,True,3380
8,habituation,LED_off_post_1s,False,282
9,habituation,LED_off_post_1s,True,2915


In [4]:
save_epoch_windows(
    windows_prepost_1s,
    pdata_root=pdata_root,
    filename="behavior_epoch_windows.h5",
    key="windows/prepost_1s"
)

[SAVED] Epoch windows: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5
[KEY] windows/prepost_1s


PosixPath('/mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_windows.h5')

In [5]:
cache = Path(pdata_root) / "_cache"
cache.mkdir(parents=True, exist_ok=True)

encoder_metrics_file = cache / "behavior_epoch_metrics.h5"

encoder_epoch_df.to_hdf(
    encoder_metrics_file,
    key="encoder/window_metrics_1s",
    mode="w",
    format="table"
)

print("Saved:", encoder_metrics_file)

Saved: /mnt/pdata/Classical_Conditioning/_cache/behavior_epoch_metrics.h5


In [6]:
encoder_epoch_df.groupby(["phase", "epoch_name"]).size().reset_index(name="n_windows")

,phase,epoch_name,n_windows
0,air_training,air_off_center,3380
1,air_training,air_on_center,3380
2,habituation,LED_off_center,2915
3,habituation,LED_on_center,2915
4,tone_air_training,air_off_center,1685
5,tone_air_training,air_on_center,1685
6,tone_air_training,tone_off_center,1685
7,tone_air_training,tone_on_center,1685


In [7]:
summary_speed = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        mean_speed=("mean_speed_path_cms", "mean"),
        sem_speed=("mean_speed_path_cms", lambda x: x.std() / np.sqrt(len(x))),
        n=("mean_speed_path_cms", "count")
    )
    .reset_index()
)

summary_speed

,phase,epoch_name,mean_speed,sem_speed,n
0,air_training,air_off_center,6.226388,0.183213,3380
1,air_training,air_on_center,1.705310,0.034726,3380
2,habituation,LED_off_center,1.719026,0.064177,2915
3,habituation,LED_on_center,1.400333,0.052130,2915
4,tone_air_training,air_off_center,4.651130,0.039276,1685
5,tone_air_training,air_on_center,1.015747,0.024942,1685
6,tone_air_training,tone_off_center,6.265828,0.070467,1685
7,tone_air_training,tone_on_center,0.587699,0.027133,1685


In [8]:
state_summary = (
    encoder_epoch_df
    .groupby(["phase", "epoch_name"])
    .agg(
        frac_stationary=("frac_stationary", "mean"),
        frac_forward=("frac_forward", "mean"),
        frac_backward=("frac_backward", "mean"),
        frac_low_net=("frac_low_net_movement", "mean"),
        n=("frac_stationary", "count")
    )
    .reset_index()
)

state_summary

,phase,epoch_name,frac_stationary,frac_forward,frac_backward,frac_low_net,n
0,air_training,air_off_center,0.067437,0.898347,0.034216,0.0,3380
1,air_training,air_on_center,0.595183,0.353475,0.051342,0.0,3380
2,habituation,LED_off_center,0.742896,0.226815,0.030289,0.0,2915
3,habituation,LED_on_center,0.764472,0.199824,0.035704,0.0,2915
4,tone_air_training,air_off_center,0.098344,0.845463,0.056193,0.0,1685
5,tone_air_training,air_on_center,0.690426,0.277400,0.032174,0.0,1685
6,tone_air_training,tone_off_center,0.081787,0.912386,0.005827,0.0,1685
7,tone_air_training,tone_on_center,0.838745,0.142160,0.019095,0.0,1685


In [9]:
tone_df = encoder_epoch_df[
    encoder_epoch_df["phase"] == "tone_air_training"
].copy()

tone_df.groupby("epoch_name").agg(
    mean_path_speed=("mean_speed_path_cms", "mean"),
    frac_forward=("frac_forward", "mean"),
    frac_stationary=("frac_stationary", "mean"),
    n=("mean_speed_path_cms", "count")
).reset_index()

,epoch_name,mean_path_speed,frac_forward,frac_stationary,n
0,air_off_center,4.651130,0.845463,0.098344,1685
1,air_on_center,1.015747,0.277400,0.690426,1685
2,tone_off_center,6.265828,0.912386,0.081787,1685
3,tone_on_center,0.587699,0.142160,0.838745,1685
